# Ingestion Pipeline
- second nootbook to Ingestion json to FAISS vector store

### **egyptian-civil-code.json** -> **Document objects** -> **Embeddings** -> **FAISS Vector Store** -> **Save locally**

### Install

In [13]:
%pip install sentence-transformers faiss-cpu
%pip install langchain langchain-community 
%pip install -U langchain-huggingface sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import the dependencies

In [3]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

C:\Users\AlrassPOS\AppData\Local\Temp\ipykernel_8100\1131237601.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### Load JSON file

In [4]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        articles = json.load(f)
    return articles

articles = load_json("../data/egyptian-civil-code.json")
print(f"✅ Loaded {len(articles)} articles")
print(f"✅ Sample: {articles[0]}")

✅ Loaded 1092 articles
✅ Sample: {'article_number': 1, 'article_text': 'مادة٢ ( لا يجوز إلغاء نص تشريعي إلا بتشريع لاحق ينص صراحة على هذا الإلغاء، أو يشتمل على نص يتعارض مع نص التشريع القديم ، أو ينظم من جديد الموضوع الذي سبق أن قرر قواعده ذلك التشريع .', 'article_text_en': 'Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.', 'is_repealed': False}


### convert json to document

In [5]:
def convert_json_to_docs(articles):
    docs = []

    for article in articles:
        combiner_text = f"""
            مادة {article['article_number']}
                {article['article_text']}

            article {article['article_number']}
                    {article['article_text_en']}
            """.strip()

        doc = Document(
            page_content=combiner_text,
            metadata={
                "article_number": article['article_number'],
                'is_repealed':    article['is_repealed'],
                "source":         f"Article {article['article_number']}"
            }
        )

        docs.append(doc)

        return docs

docs = convert_json_to_docs(articles)
print(f"✅ Created {len(docs)} documents")
print(f"\n=== Sample Document ===")
print(docs[0].page_content)
print(f"Metadata: {docs[0].metadata}")
    

✅ Created 1 documents

=== Sample Document ===
مادة 1
                مادة٢ ( لا يجوز إلغاء نص تشريعي إلا بتشريع لاحق ينص صراحة على هذا الإلغاء، أو يشتمل على نص يتعارض مع نص التشريع القديم ، أو ينظم من جديد الموضوع الذي سبق أن قرر قواعده ذلك التشريع .

            article 1
                    Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.
Metadata: {'article_number': 1, 'is_repealed': False, 'source': 'Article 1'}


### Load Embedding model

In [6]:
def load_embeddings(model_name):
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embeddings = load_embeddings("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding model loaded")

C:\Users\AlrassPOS\AppData\Local\Temp\ipykernel_8100\288004375.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)
c:\Users\AlrassPOS\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\AlrassPOS\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your

✅ Embedding model loaded


### crreate FAISS Vector Store

In [7]:
print("⏳ Creating vector store...")

vector_store = FAISS.from_documents(docs, embeddings)

print("✅ Vector store created")

⏳ Creating vector store...
✅ Vector store created


### Save Vector Store

In [8]:
vector_store_path = Path("../data/vector_store")
vector_store.save_local(str(vector_store_path))

print(f"✅ Vector store saved to {vector_store_path}")

✅ Vector store saved to ..\data\vector_store


### Load and test

In [10]:
vector_store_loaded = FAISS.load_local(
    vector_store_path, 
    embeddings,
    allow_dangerous_deserialization=True
    )

question_ar = "ما هي شروط صحة العقد؟"
question_en = "What are the conditions of a valid contract?"

results_en = vector_store_loaded.similarity_search(question_en, k=3)
print(f"\n=== English Query: {question_en} ===")
print("========= Top 3 Results ==========")

for i, result in enumerate(results_en):
    print(f"\nResult {i+1}:")
    print(result.page_content)
    print(f"Metadata: {result.metadata}")


print("\n\n")

results_ar = vector_store_loaded.similarity_search(question_ar, k=3)
print(f"\n=== Arabic Query: {question_ar} ===")
print("========= Top 3 Results ==========")

for i, result in enumerate(results_ar):
    print(f"\nResult {i+1}:")
    print(result.page_content)
    print(f"Metadata: {result.metadata}")



=== English Query: What are the conditions of a valid contract? ===
========= Top 3 Results ==========

Result 1:
مادة 1
                مادة٢ ( لا يجوز إلغاء نص تشريعي إلا بتشريع لاحق ينص صراحة على هذا الإلغاء، أو يشتمل على نص يتعارض مع نص التشريع القديم ، أو ينظم من جديد الموضوع الذي سبق أن قرر قواعده ذلك التشريع .

            article 1
                    Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.
Metadata: {'article_number': 1, 'is_repealed': False, 'source': 'Article 1'}




=== Arabic Query: ما هي شروط صحة العقد؟ ===
========= Top 3 Results ==========

Result 1:
مادة 1
                مادة٢ ( لا يجوز إلغاء نص تشريعي إلا بتشريع لاحق ينص صراحة 

In [12]:
# شوف أول 5 articles
for a in articles[:5]:
    print(f"Article {a['article_number']}:")
    print(f"  Arabic length: {len(a['article_text'])}")
    print(f"  English length: {len(a['article_text_en'])}")
    print(f"  Arabic sample: {a['article_text'][:100]}")
    print(f"  English sample: {a['article_text_en'][:100]}")
    print()

Article 1:
  Arabic length: 181
  English length: 384
  Arabic sample: مادة٢ ( لا يجوز إلغاء نص تشريعي إلا بتشريع لاحق ينص صراحة على هذا الإلغاء، أو يشتمل على نص يتعارض مع
  English sample: Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the ab

Article 2:
  Arabic length: 76
  English length: 238
  Arabic sample: مادة٣ ( تحسب المواعيد بالتقويم الميلادي، ما لم ينص هذا القانون على غير ذلك .
  English sample: A provision of a law can only be repealed by a subsequent law expressly providing for such repeal, o

Article 3:
  Arabic length: 79
  English length: 123
  Arabic sample: مادة٤ ( من استعمل حقه استعمالاً مشروعاً لا يكون مسئولاً عما ينشأ عن ذلك من ضرر.
  English sample: Periods of limitation will be calculated according to the Gregorian calendar, unless expressly provi

Article 4:
  Arabic length: 257
  English length: 94
  Arabic sample: مادة٥( يكون استعمال الحق غير مشروع في الأحوال الآتية: )أ) إذا لم يقصد به سوى الإضرار بالغير. (ب) إذا
